# 05 — Pitch and Periodicity Proxies

## Kaynak ve ayarlar

Aynı gerçek kuş kaydı üzerinde pYIN ile F0 çıkarılır. Pitch arama aralığı 150–8000 Hz’dir; bu aralık dışındaki kaynaklar ayrıca ayarlanmalıdır.

In [ ]:
from pathlib import Path
import sys, json, csv, hashlib, platform, importlib.metadata
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display
CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
sys.path.insert(0,str(ROOT/"src"))
from audio.real_sample import load_robin, ATTRIBUTION, SHA256
from audio.features import frame_features, pitch_features
EXPERIMENT_ID = "05_pitch_periodicity"
FIGURES = ROOT/"results"/"figures"/EXPERIMENT_ID
TABLES = ROOT/"results"/"tables"/EXPERIMENT_ID
EXPERIMENT = ROOT/"experiments"/EXPERIMENT_ID
for folder in [FIGURES,TABLES,EXPERIMENT]: folder.mkdir(parents=True,exist_ok=True)
y,sr,wav = load_robin(ROOT)
print(ATTRIBUTION)
print(f"sr={sr} Hz, duration={len(y)/sr:.4f} s")
display(Audio(y,rate=sr,normalize=False))


## F0, voicing ve periodicity

F0 temel frekans tahminidir. pYIN voicing olasılığı, periyodik kaynak hipotezinin algoritmaya bağlı göstergesidir; kalibre edilmiş doğruluk olasılığı değildir. RMS > −50 dBFS, voiced flag ve probability ≥0.8 birlikte pitch maskesi oluşturur. Maskelenen frame’ler boş bırakılır.

ACF skoru, tahmin edilen periyotta iki kaydırılmış frame parçasının normalize korelasyonudur. Bir periodicity proxy’sidir; CREPE çıktısıyla aynı değildir. Skor tahmini F0’a bağlı olduğundan bağımsız doğrulama veya ground truth değildir.

In [ ]:
settings={"frame_length":2048,"hop_length":256,"fmin":150,"fmax":8000,
          "threshold_db":-50,"probability_threshold":0.8}
features=pitch_features(y,sr,**settings)
mask=features["reliable_pitch"]
metrics={"frames":len(mask),"reliable_pitch_frames":int(mask.sum()),
         "reliable_pitch_fraction":float(mask.mean()),
         "median_f0_reliable_hz":float(np.nanmedian(features["f0_hz"])) if mask.any() else None,
         "mean_voicing_probability":float(np.mean(features["voiced_probability"]))}
fig,axes=plt.subplots(3,1,figsize=(11,8),sharex=True)
axes[0].plot(features["time_s"],features["f0_hz"]);axes[0].set(ylabel="F0 (Hz)")
axes[1].plot(features["time_s"],features["voiced_probability"]);axes[1].set(ylabel="pYIN voicing",ylim=(0,1.05))
axes[1].axhline(0.8,color="gray",linestyle="--")
axes[2].plot(features["time_s"],features["acf_periodicity_proxy"]);axes[2].set(ylabel="ACF proxy",xlabel="Zaman (s)",ylim=(-1,1.05))
plt.tight_layout();fig.savefig(FIGURES/"pitch_and_periodicity.svg",bbox_inches="tight");plt.show()


## Bilinen sinyal kontrolleri

440 Hz sinüs, beyaz gürültü ve sessizlik üzerinde aynı maskenin davranışı ölçülür. Testler gerçek kuş kaydındaki pitch’in doğru olduğunu kanıtlamaz; noise üzerinde yanlış pozitifler ayrıca raporlanır.

In [ ]:
test_sr=22050
t=np.arange(test_sr)/test_sr
rng=np.random.default_rng(42)
tests={"tone_440":0.2*np.sin(2*np.pi*440*t),"noise":0.05*rng.standard_normal(len(t)),"silence":np.zeros_like(t)}
checks={}
for name,signal in tests.items():
    result=pitch_features(signal,test_sr,**settings)
    m=result["reliable_pitch"]
    checks[name]={"reliable_fraction":float(m.mean()),
                  "median_f0_hz":float(np.nanmedian(result["f0_hz"])) if m.any() else None}
assert checks["tone_440"]["reliable_fraction"]>0.8
assert abs(checks["tone_440"]["median_f0_hz"]-440)<10
assert checks["silence"]["reliable_fraction"]==0
(EXPERIMENT/"diagnostics.json").write_text(json.dumps(checks,indent=2),encoding="utf-8")
print(checks)


## Çıktılar

Frame F0, voicing probability ve ACF proxy CSV’ye yazılır. Tanımsız F0 ve ACF alanları boştur. Bu inceleme pitch/periodicity kontrolünü bir modele uygulamaz.

In [ ]:
with (TABLES/"frame_features.csv").open("w",newline="",encoding="utf-8") as f:
    writer=csv.writer(f)
    writer.writerow(features.keys())
    for values in zip(*features.values()):
        writer.writerow([v.item() if isinstance(v,np.generic) and np.isfinite(v) else ("" if isinstance(v,(float,np.floating)) and not np.isfinite(v) else v) for v in values])
run={"experiment_id":EXPERIMENT_ID,"time_utc":datetime.now(timezone.utc).isoformat(),
     "source":ATTRIBUTION,"source_ogg_sha256":SHA256,"python":platform.python_version(),
     "platform":platform.system(),"settings":settings,"metrics":metrics,
     "packages":{p:importlib.metadata.version(p) for p in ["numpy","librosa","soundfile","matplotlib"]}}
(EXPERIMENT/"run.json").write_text(json.dumps(run,ensure_ascii=False,indent=2,allow_nan=False),encoding="utf-8")
print(metrics)
